# Demo 08 — Final Project Walkthrough: Customer Churn

**Machine Learning with scikit-learn — Session 08**

โน้ตบุ๊คนี้คือ "ฉบับอาจารย์" ของโปรเจ็คจบคอร์ส — เดินครบทั้ง 8 ขั้นตอนด้วยความเร็วสไลด์
ตัวคุณเองจะทำฉบับเต็ม (พร้อม EDA/error analysis ของตัวเอง) ใน **Lab 08**

## Step 0 — Load + the TotalCharges trap

In [1]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")   # ' ' → NaN 11 จุด
df.shape

(7043, 21)

## Step 1 — EDA highlights (ของจริง 3 ข้อ)

In [2]:
churn = (df["Churn"] == "Yes").astype(int)
print("churn rate:", round(churn.mean(), 3))
df.assign(churn=churn).groupby("Contract")["churn"].agg(["mean", "count"]).round(3)

churn rate: 0.265


,mean,count
Contract,,
Month-to-month,0.427,3875
One year,0.113,1473
Two year,0.028,1695


Month-to-month 42.7% vs Two-year 2.8% — ประเภทสัญญาคือพระเอกของเรื่องนี้

In [3]:
df.assign(churn=churn).groupby("InternetService")["churn"].mean().round(3)

InternetService
DSL            0.190
Fiber optic    0.419
No             0.074
Name: churn, dtype: float64

In [4]:
df.assign(churn=churn).groupby("churn")[["tenure", "MonthlyCharges"]].mean().round(1)

,tenure,MonthlyCharges
churn,,
0,37.6,61.3
1,18.0,74.4


## Step 2 — Split FIRST + pipeline (ไม่มี leakage แม้แต่บรรทัดเดียว)

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X = df.drop(columns=["customerID", "Churn"])
y = churn
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
cat_cols = [c for c in X.columns if c not in num_cols]

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_cols)])

## Step 3-5 — Baseline → model zoo → (CV, ไม่แตะ test)

In [6]:
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

skf = StratifiedKFold(5, shuffle=True, random_state=42)
zoo = [
    ("Dummy", DummyClassifier(strategy="most_frequent")),
    ("RandomForest", RandomForestClassifier(random_state=42, n_jobs=-1)),
    ("HistGradBoost", HistGradientBoostingClassifier(random_state=42)),
    ("LogisticRegression", LogisticRegression(max_iter=2000)),
]
for name, m in zoo:
    pipe = Pipeline([("pre", pre), ("m", m)])
    auc = cross_val_score(pipe, Xtr, ytr, cv=skf, scoring="roc_auc").mean()
    print(f"{name:20s} CV AUC={auc:.3f}")

Dummy                CV AUC=0.500


RandomForest         CV AUC=0.818


HistGradBoost        CV AUC=0.838


LogisticRegression   CV AUC=0.846


LogisticRegression ธรรมดา ๆ ชนะ (0.846) — ฟีเจอร์สะอาด + linear separable พอดี แถมอ่านสัมประสิทธิ์ได้

## Step 6 — Threshold is a BUSINESS decision

In [7]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

pipe = Pipeline([("pre", pre), ("m", LogisticRegression(max_iter=2000))]).fit(Xtr, ytr)
proba = pipe.predict_proba(Xte)[:, 1]
print("test AUC:", round(roc_auc_score(yte, proba), 3))
print()
for th in [0.35, 0.50, 0.65]:
    yp = (proba >= th).astype(int)
    print(f"threshold {th:.2f}: recall={recall_score(yte, yp):.2f}  precision={precision_score(yte, yp):.2f}")

test AUC: 0.842

threshold 0.35: recall=0.71  precision=0.54
threshold 0.50: recall=0.56  precision=0.66
threshold 0.65: recall=0.30  precision=0.75


## Step 7 — Error analysis: what drives churn

In [8]:
feature_names = pipe[:-1].get_feature_names_out()
coefs = pd.Series(pipe[-1].coef_[0], index=[n.split("__")[1] for n in feature_names])
coefs.reindex(coefs.abs().sort_values(ascending=False).index)[:8].round(2)

tenure                        -1.23
Contract_Two year             -0.76
InternetService_Fiber optic    0.64
InternetService_DSL           -0.64
Contract_Month-to-month        0.59
MonthlyCharges                -0.58
TotalCharges                   0.51
PaperlessBilling_No           -0.33
dtype: float64

อ่านเป็นประโยคธุรกิจได้เลย:
"tenure ยิ่งสั้นยิ่งเสี่ยง · month-to-month เสี่ยงกว่าสัญญาปี · ลูกค้า fiber เสี่ยงผิดปกติสูง"

## Step 8 — Final word: test ONCE

In [9]:
yp = (proba >= 0.5).astype(int)
print(f"CV AUC 0.846 → test AUC {roc_auc_score(yte, proba):.3f}  (gap จิ๋วๆ — ซื่อสัตย์ดี)")
print(f"recall {recall_score(yte, yp):.2f} | f1 {f1_score(yte, yp):.3f} | accuracy {(yte == yp).mean():.3f}")

CV AUC 0.846 → test AUC 0.842  (gap จิ๋วๆ — ซื่อสัตย์ดี)
recall 0.56 | f1 0.604 | accuracy 0.806


---
✅ **จบ Demo 08** — ต่อไปคือฉบับของคุณเอง: **Lab 08** ใน `excercises/08.final_project_starter.ipynb`
เชลย์ฉบับเต็มอยู่ที่ `excercises/solutions/08.final_project_solution.ipynb` (อย่าแอบดูก่อนทำนะครับ 😄)